In [25]:
from dataclasses import dataclass
import portion as P
import interval_logic as il

@dataclass
class Contract:
    assumption: il.Expression
    guarantee: il.Expression

    def __str__(self) -> str:
        return f"\\ {self.assumption} | {self.guarantee} \\"

    def merge(self, other: 'Contract') -> 'Contract':
        """Merge two contracts by combining their assumptions and guarantees."""
        return Contract(il.And((self.assumption, other.assumption)),
                        il.Or((il.And((self.guarantee, other.guarantee)),
                              il.Not(il.And((self.assumption, other.assumption))))))

    def separate(self, other: 'Contract') -> 'Contract':
        """Separate the contract into its assumption and guarantee."""
        return Contract(il.Or((il.And((self.assumption, other.guarantee)),
                               il.Not(il.And((self.guarantee, other.assumption))))),
                        il.And((self.guarantee, other.assumption)))
    
    def simplify(self) -> 'Contract':
        """Simplify the contract by removing redundant assumptions and guarantees."""
        return Contract(self.assumption.simplify(), self.guarantee.simplify())

    def is_consistent(self) -> bool:
        """Check if the contract is consistent (i.e., the assumption implies the guarantee)."""
        return il.Implies(self.assumption, self.guarantee).is_tautology()

def eqprint(*args):
    print(*args, sep="\n= ", end="\n\n")

In [30]:
print('Step 1: Initial Contract')
eqprint(C_cmt := Contract(il.Bool(True), il.Bool(True)))

print('Step 2: Requirement Contract')
eqprint(C_req1 := Contract(il.Bool(True), il.Exists(P.from_string('[1, 2)', float), il.Signal('j').equals(1))))

print('Step 3: Merged Contract')
eqprint(C_cand1 := C_cmt.merge(C_req1),
        C_cand1.simplify())

# print('Step 4a: Consistency Check')
# if C_cand1.is_consistent():
#     print("The contract is consistent.")


Step 1: Initial Contract
\ ⊤ | ⊤ \

Step 2: Requirement Contract
\ ⊤ | ∃[1.0,2.0): j = 1 \

Step 3: Merged Contract
\ ⊤ ∧ ⊤ | ⊤ ∧ (∃[1.0,2.0): j = 1) ∨ ¬(⊤ ∧ ⊤) \
= \ ⊤ | ∃[1.0,2.0): j = 1 \



¬(∃[1.0,2.0): j = 1)
